<a href="https://colab.research.google.com/github/PeroronShine/education_fefu_2/blob/main/Park_math/%D0%B2%D1%8B%D1%87%D0%BC%D0%B5%D1%826.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import numpy as np

def generate_well_conditioned_integer_matrix(n, max_int=10, cond_threshold=1e3, max_trials=1000):
    # что det != 0 и cond < cond_threshold.
    for trial in range(max_trials):
        A = np.random.randint(-max_int, max_int + 1, size=(n, n))

        det = np.linalg.det(A)
        if abs(det) < 1e-8:
            continue

        cond = np.linalg.cond(A)
        # Проверяем число обусловленности
        if cond < cond_threshold:
            return A, cond

    raise RuntimeError(f"Не удалось сгенерировать подходящую матрицу за {max_trials} попыток.")

def gauss_full_pivot_with_output(A, b):
    n = len(b)
    A = A.astype(float).copy()
    b = b.astype(float).copy()

    # Перестановки строк и столбцов
    row_perm = np.arange(n)
    col_perm = np.arange(n)

    print("\n" + "="*60)
    print("НАЧАЛО МЕТОДА ГАУССА С ПОЛНЫМ ВЫБОРОМ ГЛАВНОГО ЭЛЕМЕНТА")
    print("="*60)

    print("\nРасширенная матрица [A | b]:")
    Ab = np.hstack([A, b.reshape(-1, 1)])
    print(Ab)

    for k in range(n - 1):
        print(f"\n--- ШАГ {k+1} (k = {k}) ---")

        # Найдём максимальный по модулю элемент в подматрице A[k:, k:]
        sub_matrix = A[k:, k:]
        i_max_rel, j_max_rel = np.unravel_index(np.abs(sub_matrix).argmax(), sub_matrix.shape)
        i_max = i_max_rel + k
        j_max = j_max_rel + k
        pivot_val = A[i_max, j_max]
        print(f"Максимальный элемент в подматрице A[{k}:, {k}:]: |A[{i_max}, {j_max}]| = |{pivot_val:.6f}|")
        if abs(pivot_val) < 1e-12:
            raise np.linalg.LinAlgError("Матрица вырождена или почти вырождена на шаге k={k}.")

        # Перестановка строк
        if i_max != k:
            print(f"Перестановка строк {k} и {i_max}")
            A[[k, i_max]] = A[[i_max, k]]
            b[[k, i_max]] = b[[i_max, k]]
            row_perm[[k, i_max]] = row_perm[[i_max, k]]
        else:
            print(f"Строка {k} остаётся на месте.")

        # Перестановка столбцов
        if j_max != k:
            print(f"Перестановка столбцов {k} и {j_max}")
            A[:, [k, j_max]] = A[:, [j_max, k]]
            col_perm[[k, j_max]] = col_perm[[j_max, k]]
        else:
            print(f"Столбец {k} остаётся на месте.")

        print(f"Текущая перестановка столбцов (новый порядок переменных): {col_perm}")

        # --- ПРЯМОЙ ХОД ---
        print(f"Преобразование строк ниже {k} для обнуления элементов под A[{k}, {k}] = {A[k, k]:.6f}")

        for i in range(k + 1, n):
            if A[i, k] != 0:
                factor = A[i, k] / A[k, k]
                A[i, k:] -= factor * A[k, k:]
                b[i] -= factor * b[k]
                print(f"  Строка {i} = Строка {i} - ({factor:.6f}) * Строка {k}")
            else:
                print(f"  Элемент A[{i}, {k}] уже равен 0, преобразование не требуется.")

        print("\nМатрица A и вектор b после шага {k+1}:")
        Ab_current = np.hstack([A, b.reshape(-1, 1)])
        print(Ab_current)

    print("\n--- КОНЕЦ ПРЯМОГО ХОДА ---")
    print("Матрица приведена к верхнетреугольному виду.")
    print("Расширенная матрица [U | c]:")
    print(Ab_current)

    print("\n" + "="*60)
    print("ОБРАТНЫЙ ХОД")
    print("="*60)
    # Решение для переставленного вектора x_perm
    x_perm = np.zeros(n)
    for i in range(n - 1, -1, -1): # от n-1 до 0 включительно
        sum_ax = 0
        for j in range(i + 1, n):
            sum_ax += A[i, j] * x_perm[j]
        if A[i, i] == 0:
            raise ZeroDivisionError(f"Диагональный элемент A[{i}, {i}] равен 0. Система не имеет единственного решения")
        x_perm[i] = (b[i] - sum_ax) / A[i, i]
        print(f"x_perm[{i}] = (b[{i}] - sum_ax) / A[{i}, {i}] = ({b[i]:.6f} - {sum_ax:.6f}) / {A[i, i]:.6f} = {x_perm[i]:.6f}")

    print(f"\nРешение в перестановке столбцов: x_perm = {x_perm}")

    x = np.zeros(n)
    for i in range(n):
        x[col_perm[i]] = x_perm[i]

    print(f"Используем перестановку столбцов {col_perm} для восстановления исходного порядка")
    print(f"Итоговое решение x = {x}")
    return x

In [38]:
n = 4

A, cond_num = generate_well_conditioned_integer_matrix(n, max_int=5, cond_threshold=500)

print(f"Матрица A ({n}x{n}) сгенерирована")
print(f"Определитель: {np.linalg.det(A):.6f}")
print(f"Число обусловленности: {cond_num:.2f}")

print("\nМатрица A:")
print(A)

Матрица A (4x4) сгенерирована
Определитель: 35.000000
Число обусловленности: 66.01

Матрица A:
[[ 3 -5 -2  2]
 [ 4  2 -4 -3]
 [-1 -4 -3  1]
 [-3 -5  3  4]]


In [39]:
print("\nМатрица A:")
print(A)

x_true = np.random.randn(n)

#x_true = [2, 6, 1, 1]
b = A @ x_true

#b = np.random.randn(n)
print(f"\nИзвестное решение x_true = {x_true}")

print("\nВектор b:")
print(b)

x_sol = gauss_full_pivot_with_output(A, b)

print(f"\nИзвестное решение x_true = {x_true}")

print("\n" + "="*60)
print("ПРОВЕРКА РЕШЕНИЯ")
print("="*60)

residual_norm = np.linalg.norm(A @ x_sol - b)

error_norm = np.linalg.norm(x_sol - x_true)

print(f"Невязка ||A @ x_sol - b||: {residual_norm:.2e}")
print(f"Ошибка ||x_sol - x_true||: {error_norm:.2e}")

if error_norm < 1e-15 and residual_norm < 1e-15:
    print("Решение найдено точно (в пределах погрешности вычислений)")
else:
    print("Решение найдено, но возможно наличие ошибки или неустойчивости")


Матрица A:
[[ 3 -5 -2  2]
 [ 4  2 -4 -3]
 [-1 -4 -3  1]
 [-3 -5  3  4]]

Известное решение x_true = [-1.02048879  0.06004196 -2.28287116 -0.53857247]

Вектор b:
[ 0.12692119  6.78533077  7.09036196 -6.24164676]

НАЧАЛО МЕТОДА ГАУССА С ПОЛНЫМ ВЫБОРОМ ГЛАВНОГО ЭЛЕМЕНТА

Расширенная матрица [A | b]:
[[ 3.         -5.         -2.          2.          0.12692119]
 [ 4.          2.         -4.         -3.          6.78533077]
 [-1.         -4.         -3.          1.          7.09036196]
 [-3.         -5.          3.          4.         -6.24164676]]

--- ШАГ 1 (k = 0) ---
Максимальный элемент в подматрице A[0:, 0:]: |A[0, 1]| = |-5.000000|
Строка 0 остаётся на месте.
Перестановка столбцов 0 и 1
Текущая перестановка столбцов (новый порядок переменных): [1 0 2 3]
Преобразование строк ниже 0 для обнуления элементов под A[0, 0] = -5.000000
  Строка 1 = Строка 1 - (-0.400000) * Строка 0
  Строка 2 = Строка 2 - (0.800000) * Строка 0
  Строка 3 = Строка 3 - (1.000000) * Строка 0

Матрица A и вект